In [42]:
import tqdm
import pickle
import warnings
import numpy as np
import pandas as pd
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from data import *
from model import *
from utils import *
from recourse_model import LAROAR, ROAR, RecourseCost

warnings.filterwarnings('ignore')

In [43]:
def append_res(d, alg, seed, alpha, lamb, i, x_0, theta_0, beta, x_r, theta_r, p, theta_p, J_r, J_c, robustness, consistency):
    d['alg'].append(alg)
    d['seed'].append(seed)
    d['alpha'].append(alpha)
    d['lambda'].append(lamb)
    d['i'].append(i)
    d['x_0'].append(x_0.round(4))
    d['theta_0'].append(theta_0.round(4))
    d['beta'].append(beta)
    d['x_r'].append(x_r.round(4))
    d['theta_r'].append(theta_r.round(4))
    d['p'].append(p)
    d['theta_p'].append(theta_p.round(4))
    d['J_r'].append(J_r)
    d['J_c'].append(J_c)
    d['robustness'].append(robustness)
    d['consistency'].append(consistency)

In [44]:
def recourse_model_runner(seed, X: np.ndarray, recourse_model1: LAROAR, recourse_model2: ROAR, predictions: List, X_train: np.ndarray, base_model: LR | NN):    
    alpha = recourse_model1.alpha
    lamb = recourse_model1.lamb
    
    results_opt = {'alg': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'theta_0': [], 'beta': [], 'x_r': [], 'theta_r': [], 'p': [], 'theta_p': [], 'J_r': [], 'J_c': [], 'robustness': [], 'consistency': []}
    results_roar = deepcopy(results_opt)
    
    n = len(X)
    
    weights_0, bias_0 = recourse_model1.weights, recourse_model1.bias
    theta_0 = np.hstack((weights_0, bias_0))
    for i in tqdm.trange(n, desc=f'Eval alpha={alpha}; lambda={lamb}', colour='#0091ff'):
        x_0 = X[i]
        J = RecourseCost(x_0, lamb)
        
        for p, prediction in enumerate(predictions):
            weights_p, bias_p = prediction[:-1], prediction[[-1]]
            theta_p = (weights_p, bias_p)
        
            # robustness
            x_r = recourse_model1.get_recourse(x_0, beta=1.)
            weights_r, bias_r = recourse_model1.calc_theta_adv(x_r)
            opt_rob = J(x_r, weights_r, bias_r)
            
            # consistency
            x_c = recourse_model1.get_recourse(x_0, beta=0., theta_p=theta_p)
            opt_con = J(x_c, weights_p, bias_p)
            
            # OPT
            betas = np.arange(0., 1.01, 0.01).round(4)
            alg_num = 0
            for beta in betas:
                x = recourse_model1.get_recourse(x_0, beta=beta, theta_p=theta_p)
                weights_r, bias_r = recourse_model1.calc_theta_adv(x)
                theta_r = np.hstack((weights_r, bias_r))
                
                J_r = J(x, weights_r, bias_r)
                J_c = J(x, weights_p, bias_p)
                rob = J_r - opt_rob
                con = J_c - opt_con
                
                append_res(results_opt, 'OPT', seed, alpha, lamb, i, x_0, theta_0, beta, x, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                
                # convex combination
                for c in np.arange(0., 1.01, 0.01).round(2):
                    x_b = c*x + (1-c)*x_c
                    
                    weights_r, bias_r = recourse_model1.calc_theta_adv(x_b)
                    theta_r = np.hstack((weights_r, bias_r))
                    
                    J_r = J(x_b, weights_r, bias_r)
                    J_c = J(x_b, weights_p, bias_p)
                    rob = J_r - opt_rob
                    con = J_c - opt_con
                    
                    append_res(results_opt, f'OPT{alg_num}', seed, alpha, lamb, i, x_0, theta_0, c, x_b, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                    
                    x_b = c*x_r + (1-c)*x
                    weights_r, bias_r = recourse_model1.calc_theta_adv(x_b)
                    theta_r = np.hstack((weights_r, bias_r))
                    
                    J_r = J(x_b, weights_r, bias_r)
                    J_c = J(x_b, weights_p, bias_p)
                    rob = J_r - opt_rob
                    con = J_c - opt_con
                    
                    append_res(results_opt, f'OPT{alg_num+1}', seed, alpha, lamb, i, x_0, theta_0, c, x_b, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                
                alg_num += 2
            
            # ROAR
            x_r2, _ = recourse_model2.get_recourse(x_0)
            weights_r2, bias_r2 = recourse_model1.calc_theta_adv(x_r2)
            theta_r2 = np.hstack((weights_r2, bias_r2))
            J_r2 = J(x_r2, weights_r2, bias_r2)
            J_c2 = J(x_r2, weights_p, bias_p)
            rob2 = J_r2 - opt_rob
            con2 = J_c2 - opt_con
            
            append_res(results_roar, 'ROAR', seed, alpha, lamb, i, x_0, theta_0, 1., x_r2, theta_r2, p, np.hstack(theta_p), J_r2[0], J_c2[0], rob2[0], con2[0])
        
    df_hist = pd.concat((pd.DataFrame(results_opt), pd.DataFrame(results_roar)))
    df_results = df_hist.groupby(['alg', 'beta', 'p'], as_index=False).mean(True)
        
    return df_results

In [45]:
def recourse_tradeoff(params: dict, seeds):
    if params['data'] in ['correction', 'german']:
        data_model = CorrectionShift("../datasets/german.csv", "../datasets/corrected_german.csv", seed=0)
    elif params['data'] in ['temporal', 'business']:
        data_model = TemporalShift("../datasets/SBAcase.11.13.17.csv", seed=0)
    elif params['data'] in ['geospatial', 'student']:
        data_model = GeospatialShift("../datasets/student-por.csv", seed=0)
    elif params['data'] in ['synthetic', 'simulated']:
        data_model = SyntheticData(alpha=1.5, beta=0, n=1000, seed=0)
    alpha = params['alpha']
    
    results = []
        
    for seed in seeds:
        data1, data2 = data_model.get_data(seed)
        X1_train, y1_train, X1_test, y1_test = data1

        base_model = LR()
        base_model.train(X1_train.values, y1_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        theta_0 = np.hstack((weights_0, bias_0))
        
        if seed == 0:
            theta_preds_near = np.load(f'../theta_preds/theta_preds_{params["base_model"]}_{params["data"]}.npy')
            predictions = [theta_0]
            for theta_p in theta_preds_near:
                alphas = theta_p - theta_0
                theta = theta_0 - alphas
                predictions.append(theta_p)
                predictions.append(theta)
            
        recourse_needed_X1_train = recourse_needed(base_model.predict, X1_train.values)
        recourse_needed_X1_test = recourse_needed(base_model.predict, X1_test.values)
        
        weights, bias = weights_0, bias_0
        
        recourse_model1 = LAROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )
        
        recourse_model2 = ROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )
        
        lamb = recourse_model1.choose_lambda(recourse_needed_X1_train, base_model.predict, X1_train.values)
        recourse_model1.lamb = lamb
        recourse_model2.lamb = lamb
        
        df_results = recourse_model_runner(seed, recourse_needed_X1_test, recourse_model1, recourse_model2, predictions, X1_train.values, base_model)
        results.append(df_results)

    return results

In [46]:
torch.manual_seed(0)
params = {}
# 'synthetic/simulated', 'correction/german', 'temporal/business', 'geospatial/student'
params['data'] = 'temporal'
# 'lr', 'nn
params['base_model'] = 'lr'
params['alpha'] = 0.5

seeds = range(5)

results = recourse_tradeoff(params, seeds)
df_results = pd.concat(results)

Eval alpha=0.5; lambda=1.0: 100%|██████████| 38/38 [15:29<00:00, 24.46s/it]


In [47]:
df_results.to_pickle(f'../results/tradeoff_{params["base_model"]}_{params["data"]}.pkl')